In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/Student Social Media And Mental Health Impact.csv")
df

,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score
0,21,Male,Other,Undergraduate,Facebook,Networking,4.0,134,4.5,2.2,6.7,Medium,6.8
1,23,Female,Other,Graduate,LinkedIn,Education,1.6,73,7.0,2.4,8.6,Low,7.6
2,22,Male,Canada,Undergraduate,Instagram,Entertainment,4.6,166,4.0,1.8,6.7,Medium,7.0
3,18,Male,Other,High School,Snapchat,Entertainment,7.0,220,1.0,1.7,5.4,Very High,5.3
4,24,Female,Other,Graduate,Facebook,Networking,7.5,237,1.0,1.1,5.0,Very High,4.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,18,Female,Other,High School,YouTube,Education,1.9,89,6.1,2.2,7.9,Low,7.9
4996,18,Female,Other,High School,YouTube,Education,6.6,216,2.3,1.8,6.0,Very High,4.7
4997,19,Female,India,Undergraduate,LinkedIn,Education,3.8,151,5.4,1.4,7.5,Medium,7.0
4998,18,Male,Other,High School,Instagram,Entertainment,3.8,119,4.0,1.5,6.4,Medium,6.7


## Paso 3: Almacenar la información

In [9]:
import sqlite3

nombre_base_datos = 'student_social_media_and_mental_health_impact.db'
conn = sqlite3.connect(nombre_base_datos)

df.to_sql('student_social_media_and_mental_health_impact', conn, if_exists='replace', index=False)

conn.close()

Haremos algunas consultas:
- La primera consulta será para obtener la cantidad de registros en la tabla 'student-social-media-and-mental-health-impact'.

In [11]:
conn = sqlite3.connect(nombre_base_datos)
cursor = conn.cursor()

query_1 = "SELECT COUNT(*) FROM student_social_media_and_mental_health_impact"

cursor.execute(query_1)

total_registros = cursor.fetchone()[0]

print(f"Cantidad total de registros: {total_registros}")

Cantidad total de registros: 5000


- La segunda consulta será para ver y comparar las clases (**'Female'** - **'Male'**) de la columna género (***"Gender"***) por cada categoría de nivel de estrés (***"Stress_Level"***).

In [69]:
query_2 = """
SELECT 
    Gender,
    Stress_Level,
    COUNT(*) AS total_estudiantes
FROM student_social_media_and_mental_health_impact
GROUP BY Stress_Level, Gender
ORDER BY Stress_Level;
"""

cursor.execute(query_2)
resultados = cursor.fetchall()
    
nivel_actual = None

for gender, stress_level, total_estudiantes in resultados:
    if stress_level != nivel_actual:
        if nivel_actual is not None:
            print()
        print(f"Nivel de Estrés: {stress_level}")
        nivel_actual = stress_level

    print(f"  - {gender}: {total_estudiantes} estudiantes")


Nivel de Estrés: High
  - Female: 720 estudiantes
  - Male: 720 estudiantes

Nivel de Estrés: Low
  - Female: 301 estudiantes
  - Male: 343 estudiantes

Nivel de Estrés: Medium
  - Female: 564 estudiantes
  - Male: 731 estudiantes

Nivel de Estrés: Very High
  - Female: 780 estudiantes
  - Male: 841 estudiantes


- En la tercera consulta veremos el promedio del puntaje de salud mental (***"Mental_Health_Score"***) por cada categoría de nivel de estrés (***"Stress_Level"***).

In [22]:
query_3 = """
SELECT 
    Stress_Level,
    COUNT(*) AS total_estudiantes,
    ROUND(AVG(Mental_Health_Score), 2) AS promedio_puntaje_salud_mental
FROM student_social_media_and_mental_health_impact
GROUP BY Stress_Level
ORDER BY promedio_puntaje_salud_mental DESC;
"""

cursor.execute(query_3)
resultados = cursor.fetchall()

for stress_level, total_estudiantes, promedio_puntaje_salud_mental in resultados:
    print(f"Nivel de estrés: {stress_level}")
    print(f"  - Total estudiantes: {total_estudiantes}")
    print(f"  - Promedio puntaje de salud mental: {promedio_puntaje_salud_mental}")

Nivel de estrés: Low
  - Total estudiantes: 644
  - Promedio puntaje de salud mental: 7.84
Nivel de estrés: Medium
  - Total estudiantes: 1295
  - Promedio puntaje de salud mental: 7.14
Nivel de estrés: High
  - Total estudiantes: 1440
  - Promedio puntaje de salud mental: 6.03
Nivel de estrés: Very High
  - Total estudiantes: 1621
  - Promedio puntaje de salud mental: 5.05


- La cuarta consulta se trata de agrupar a los estudiantes de cada clase de nivel de estrés (***"Stress_Level"***) por cada categoría de nivel académico (***"Academic_Level"***).

In [66]:
query_4 = """
SELECT 
    Academic_Level,
    Stress_Level,
    COUNT(*) AS total_estudiantes
FROM student_social_media_and_mental_health_impact
GROUP BY Academic_Level, Stress_Level;
"""

cursor.execute(query_4)
resultados = cursor.fetchall()

nivel_actual = None

for academic_level, stress_level, total_estudiantes in resultados:
    if academic_level != nivel_actual:
        if nivel_actual is not None:
            print()
        print(f"Nivel Académico: {academic_level}")
        nivel_actual = academic_level
    
    print(f"  - Nivel de Estrés ({stress_level}): {total_estudiantes} estudiantes")



Nivel Académico: Graduate
  - Nivel de Estrés (High): 211 estudiantes
  - Nivel de Estrés (Low): 154 estudiantes
  - Nivel de Estrés (Medium): 243 estudiantes
  - Nivel de Estrés (Very High): 310 estudiantes

Nivel Académico: High School
  - Nivel de Estrés (High): 107 estudiantes
  - Nivel de Estrés (Low): 51 estudiantes
  - Nivel de Estrés (Medium): 76 estudiantes
  - Nivel de Estrés (Very High): 216 estudiantes

Nivel Académico: Undergraduate
  - Nivel de Estrés (High): 1122 estudiantes
  - Nivel de Estrés (Low): 439 estudiantes
  - Nivel de Estrés (Medium): 976 estudiantes
  - Nivel de Estrés (Very High): 1095 estudiantes


- En nuestra quinta y última consulta veremos el promedio del uso diario de RRSS (***"Avg_Daily_Usage_Hours"***), el promedio de las horas diarias de estudio (***"Study_Hours"***), el promedio de las horas diarias de actividad física (***"Physical_Activity_Hours"***) y el promedio de las horas diarias de sueño (***"Physical_Activity_Hours"***) agrupadas por el género (***"Gender"***).

In [74]:
query_5 = """
SELECT 
    Gender,
    ROUND(AVG(Avg_Daily_Usage_Hours), 2) AS promedio_pantalla_RRSS,
    ROUND(AVG(Study_Hours), 2) AS promedio_horas_estudio,
    ROUND(AVG(Physical_Activity_Hours), 2) AS promedio_horas_actividad_fisica,
    ROUND(AVG(Sleep_Hours_Per_Night), 2) AS promedio_horas_sueno
FROM student_social_media_and_mental_health_impact
GROUP BY Gender;
"""

cursor.execute(query_5)
resultados = cursor.fetchall()

for Gender, promedio_pantalla_RRSS, promedio_horas_estudio, promedio_horas_actividad_fisica, promedio_horas_sueno in resultados:
    print(f"Género ({Gender}):")
    print(f"  - Promedio Uso RRSS diario: {promedio_pantalla_RRSS} hrs")
    print(f"  - Promedio Horas de Estudio diario: {promedio_horas_estudio} hrs")
    print(f"  - Promedio Horas de Actividad Física diario: {promedio_horas_actividad_fisica} hrs")
    print(f"  - Promedio Horas de Sueño diario: {promedio_horas_sueno}")
    
    
conn.close()

Género (Female):
  - Promedio Uso RRSS diario: 5.13 hrs
  - Promedio Horas de Estudio diario: 2.97 hrs
  - Promedio Horas de Actividad Física diario: 1.75 hrs
  - Promedio Horas de Sueño diario: 6.61
Género (Male):
  - Promedio Uso RRSS diario: 5.03 hrs
  - Promedio Horas de Estudio diario: 3.05 hrs
  - Promedio Horas de Actividad Física diario: 1.75 hrs
  - Promedio Horas de Sueño diario: 6.66
